# Lab 10: Performance Tuning & Spark UI

## Objectives
- Read and interpret execution plans (logical, physical)
- Use caching and persistence levels effectively
- Optimize joins with broadcast hints
- Understand and use Adaptive Query Execution (AQE)
- Diagnose performance issues using the Spark UI

## Exam Domains
- **Troubleshooting and Tuning DataFrame API Apps — 10%**
- **Apache Spark Architecture and Components — 20%**

## Estimates
- **Time:** ~40 minutes
- **Cost:** $2-3 (requires cluster)
- **Compute:** Single-node cluster

![Optimization & Execution Plans](../assets/diagrams/lab-10-optimization-execution-plan.png)

In [ ]:
# This lab requires a cluster (not serverless) for Spark UI access
USE_CATALOG = "spark_lab_guide"
USE_SCHEMA = "default"

spark.sql(f"USE CATALOG {USE_CATALOG}")
spark.sql(f"USE SCHEMA {USE_SCHEMA}")

taxi_df = spark.table("taxi_trips")
github_df = spark.table("github_events")

## A. Reading Execution Plans

`explain()` shows how Spark will execute a query. Understanding the plan is key to diagnosing performance issues.

In [ ]:
from pyspark.sql.functions import col, avg, count

# Build a query
query = (
    taxi_df
    .filter(col("trip_distance") > 5)
    .groupBy("PULocationID")
    .agg(
        count("*").alias("trips"),
        avg("total_amount").alias("avg_fare"),
    )
    .orderBy(col("trips").desc())
)

# Simple physical plan
print("=== Physical Plan ===")
query.explain()

In [ ]:
# Extended plan — shows all stages
print("=== Extended Plan (all stages) ===")
query.explain(True)

In [ ]:
# Formatted plan — most readable
print("=== Formatted Plan ===")
query.explain("formatted")

> **Exam Tip:** The plan reads bottom-to-top. Key operators to recognize:
> - **Scan** — reading data from source
> - **Filter** — predicate pushdown
> - **Exchange** — shuffle (data redistribution across executors)
> - **HashAggregate** — aggregation
> - **Sort** — ordering
> - **BroadcastHashJoin** — broadcast join (small table replicated to all executors)
> - **SortMergeJoin** — shuffle join (both tables shuffled)

## B. Caching and Persistence

Caching stores a DataFrame in memory (or disk) so subsequent actions don't recompute it. Useful when you reuse the same data multiple times.

In [ ]:
from pyspark import StorageLevel

# Cache in memory (default) — NOT supported on serverless
try:
    taxi_df.cache()
    # First action triggers the cache — data is computed and stored
    taxi_df.count()
    print("Data cached in memory.")
    # Second action uses the cache — much faster
    taxi_df.count()
except Exception as e:
    print(f"cache() not supported on this compute: {e}")
    print("On a cluster, cache() stores the DataFrame in memory for faster repeated access.")
    # Still run the count to keep the lab flowing
    taxi_df.count()

In [ ]:
# Persistence levels (requires a cluster — not supported on serverless)
# MEMORY_ONLY — deserialized in JVM heap
# MEMORY_AND_DISK — spill to disk if memory is full
# DISK_ONLY — only on disk
# MEMORY_ONLY_SER — serialized in memory (smaller but slower)

try:
    taxi_df.unpersist()  # Remove from cache first
    taxi_df.persist(StorageLevel.MEMORY_AND_DISK)
    taxi_df.count()  # Trigger persistence
    print("Persisted with MEMORY_AND_DISK level.")
except Exception as e:
    print(f"persist() not supported on this compute: {e}")
    print("On a cluster, persist() lets you choose the storage level.")

In [ ]:
# Check what's cached — look at Spark UI > Storage tab
try:
    print(f"Is cached: {taxi_df.is_cached}")
    taxi_df.unpersist()
    print(f"After unpersist, is cached: {taxi_df.is_cached}")
except Exception as e:
    print(f"Cache status check not available: {e}")
    print("On a cluster, use Spark UI > Storage tab to see cached DataFrames.")

> **Exam Tip:** `cache()` is shorthand for `persist(StorageLevel.MEMORY_AND_DISK)` in PySpark (not MEMORY_ONLY as in Scala). The exam may test which storage level `cache()` uses.

## C. Broadcast Joins

When joining a large table with a small table, broadcasting the small table to all executors avoids a shuffle. This is much faster than a sort-merge join.

In [ ]:
from pyspark.sql.functions import broadcast, hour, to_timestamp

# Create a small lookup table
payment_lookup = spark.createDataFrame([
    (1, "Credit Card"),
    (2, "Cash"),
    (3, "No Charge"),
    (4, "Dispute"),
    (5, "Unknown"),
], ["payment_type", "payment_label"])

# Without broadcast hint — Spark decides (may or may not broadcast)
joined_auto = taxi_df.join(payment_lookup, on="payment_type")
print("=== Auto join strategy ===")
joined_auto.explain()

In [ ]:
# With explicit broadcast hint — forces broadcast join
joined_broadcast = taxi_df.join(broadcast(payment_lookup), on="payment_type")
print("=== Broadcast join strategy ===")
joined_broadcast.explain()

# Execute to see performance difference
joined_broadcast.count()

## D. Adaptive Query Execution (AQE)

AQE re-optimizes the query plan at runtime based on actual data statistics. It can:
- Coalesce shuffle partitions (reduce small partitions)
- Switch join strategies (e.g., convert sort-merge to broadcast if one side is small)
- Optimize skewed joins

In [ ]:
# Check AQE status
def safe_conf_get(key, default="N/A"):
    try:
        return spark.conf.get(key)
    except Exception:
        return default

print(f"AQE enabled: {safe_conf_get('spark.sql.adaptive.enabled')}")
print(f"Auto broadcast threshold: {safe_conf_get('spark.sql.autoBroadcastJoinThreshold')}")
print(f"Coalesce partitions enabled: {safe_conf_get('spark.sql.adaptive.coalescePartitions.enabled')}")

In [ ]:
# Demonstrate shuffle partition coalescing with AQE
# Default shuffle partitions is 200 — AQE will reduce this at runtime
result = (
    taxi_df
    .groupBy("PULocationID", "DOLocationID")
    .agg(count("*").alias("trips"))
    .filter(col("trips") > 100)
)

# Run the query and check Spark UI > SQL tab
# AQE should show "CustomShuffleReader" with coalesced partitions
result.count()
print("Check Spark UI > SQL tab for AQE coalescing.")

## E. Partitioning Strategies

How you partition data affects parallelism, shuffle size, and query performance.

In [ ]:
# repartition — full shuffle, can increase or decrease partitions
df_8 = taxi_df.repartition(8)
try:
    print(f"repartition(8): {df_8.rdd.getNumPartitions()} partitions")
except Exception:
    print("repartition(8) applied")

# coalesce — no full shuffle, can only DECREASE partitions
df_2 = df_8.coalesce(2)
try:
    print(f"coalesce(2): {df_2.rdd.getNumPartitions()} partitions")
except Exception:
    print("coalesce(2) applied")

# repartition by column — ensures same key goes to same partition
df_by_key = taxi_df.repartition(8, "PULocationID")
try:
    print(f"repartition by PULocationID: {df_by_key.rdd.getNumPartitions()} partitions")
except Exception:
    print("repartition by PULocationID applied")

In [ ]:
# Impact on write performance — write with explicit partitioning
(
    taxi_df
    .repartition(4)
    .write
    .mode("overwrite")
    .saveAsTable("taxi_repartitioned_4")
)

print("Written with 4 partitions.")
spark.sql("DESCRIBE DETAIL taxi_repartitioned_4").select("numFiles").show()

## F. Diagnosing Performance with Spark UI

Key metrics to check in the Spark UI:

| What to Check | Where | What it Means |
|---------------|-------|---------------|
| Shuffle read/write | Stages tab | Large shuffle = potential bottleneck |
| Spill (memory) | Stages tab | Data spilling to disk = not enough memory |
| Skew | Stage details | Tasks with much more data than others |
| GC time | Executors tab | High GC = memory pressure |
| Task duration variance | Stage details | Large variance = data skew |

In [ ]:
# Run a query that creates significant shuffle to observe in Spark UI
heavy_query = (
    taxi_df
    .join(
        taxi_df.groupBy("PULocationID").agg(avg("total_amount").alias("location_avg")),
        on="PULocationID",
    )
    .filter(col("total_amount") > col("location_avg") * 2)
    .groupBy("payment_type")
    .agg(count("*").alias("above_avg_trips"))
)

heavy_query.show()
print("Check Spark UI > SQL tab > click on the query to see the DAG visualization.")

## Exam-Style Review

**Q1.** What does `cache()` do in PySpark?
- A) Immediately stores the DataFrame in memory
- B) Marks the DataFrame for caching — actual caching happens on the next action
- C) Writes the DataFrame to disk
- D) Creates a checkpoint of the DataFrame

**Q2.** What is the default storage level of `cache()` in PySpark?
- A) MEMORY_ONLY
- B) MEMORY_AND_DISK
- C) DISK_ONLY
- D) MEMORY_ONLY_SER

**Q3.** When should you use `broadcast()` in a join?
- A) When both tables are large
- B) When one table is small enough to fit in executor memory
- C) When you want to avoid caching
- D) When you need a cross join

**Q4.** What is the difference between `repartition(n)` and `coalesce(n)`?
- A) They are identical
- B) `repartition` does a full shuffle; `coalesce` avoids a full shuffle but can only decrease partitions
- C) `coalesce` is always faster
- D) `repartition` does not change partition count

**Q5.** What does "Exchange" mean in a Spark execution plan?
- A) Data conversion between formats
- B) A shuffle — data is redistributed across the cluster
- C) Communication between driver and executor
- D) Swapping data between memory and disk

**Q6.** What does Adaptive Query Execution (AQE) do?
- A) Caches frequently used queries
- B) Re-optimizes the query plan at runtime based on actual data statistics
- C) Parallelizes query execution across multiple clusters
- D) Pre-compiles queries for faster execution

### Answers
- **Q1: B** — `cache()` is lazy. It marks the DataFrame for caching, but data is only cached when the next action executes.
- **Q2: B** — In PySpark, `cache()` uses `MEMORY_AND_DISK`. In Scala, it uses `MEMORY_ONLY`.
- **Q3: B** — Broadcast join is optimal when one table is small enough to replicate to all executors, avoiding a shuffle.
- **Q4: B** — `repartition()` triggers a full shuffle and can increase or decrease partitions. `coalesce()` avoids a full shuffle but can only decrease.
- **Q5: B** — Exchange in the plan means a shuffle operation where data is redistributed across executors.
- **Q6: B** — AQE optimizes at runtime: coalescing partitions, switching join strategies, and handling skew.

## Key Takeaways
- `explain(True)` shows all plan stages — read bottom-to-top
- `cache()` is lazy (PySpark default: MEMORY_AND_DISK); `persist()` allows custom storage levels
- `broadcast()` avoids shuffle for small-large joins — always broadcast the small table
- AQE re-optimizes at runtime: coalescing partitions, switching join strategies
- `repartition()` = full shuffle (increase/decrease); `coalesce()` = no shuffle (decrease only)
- Spark UI is your diagnostic tool: check shuffle size, spill, task skew, and GC time

**Congratulations!** You have completed all 10 labs. Run `python scripts/cleanup.py` to remove all resources.

In [ ]:
# Final cleanup
spark.sql("DROP TABLE IF EXISTS taxi_repartitioned_4")
spark.sql("DROP VIEW IF EXISTS taxi_summary")
try:
    taxi_df.unpersist()
except Exception:
    pass
print("Lab 10 cleanup complete.")
print("Run 'python scripts/cleanup.py' to remove all catalog resources.")